# Explore Elife Feature Counts

Andrew E. Davidson  
aedaivds@ucsc.edu 2/18/25  

Copyright (c) 2020-2023, Regents of the University of California All rights reserved. https://polyformproject.org/licenses/noncommercial/1.0.0  

**ref :**
 - intraExtraRNA_POC/jupyterNotebooks/elife/elifeBinaryRandomForestResults.ipynb
 - deconvolutionAnalysis/tempus/jupyterNotebooks/elifeLungRandomForestPredictions.ipynb
 - intraExtraRNA_POC/jupyterNotebooks/elife/gan/clusterAnalysis.ipynb
 - intraExtraRNA_POC/jupyterNotebooks/elife/createGTEx_TCGA_elifeDataSets.ipynb

**Abstract:** 

We used the Lung Binary classifier trained in elifeBinaryRandomForestResults.ipynb to make predict if the tempus Undiluted and Control samples where either Lung cancer or a healthy control. The hyperparameter tunning results using k-fold validation was  0.7515873015873017. The AUC when we trained on all the samples was 1.00

```
HUGO_Genes
['FPR3', 'CSF3', 'SLAMF8', 'ENTPD2', 'MAGEE1', 'PCAT19', 'GRIP2', 'PTGIR', 'RND1', 'CHRNB1']

elifeGenes
['ENSG00000158714.11', 'ENSG00000144596.13', 'ENSG00000054179.12', 'ENSG00000172602.11', 'ENSG00000170175.11', 'ENSG00000108342.13', 'ENSG00000267107.9', 'ENSG00000160013.9', 'ENSG00000187474.5', 'ENSG00000198934.5']
```

When we used this model make predictions on Tempus Undiluted and Control samples. The model predicted all the samples where healthy control. Looking at the Tempus count data, almost all values where zero. We never looked at the actual count data from the elife data.

In general the elife data does not cluster


In [1]:
import ipynbname

# use display() to print an html version of a data frame
# useful if dataFrame output is not generated by last like of cell
from IPython.display import display

import joblib
# import math
#import numpy as np
import os
import pandas as pd
# pd.set_option('display.max_rows', None)

#from sklearn.ensemble        import RandomForestClassifier

import sys

notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

import logging
# loglevel = "INFO"
loglevel = "WARN"
# logFMT = "%(asctime)s %(levelname)s [thr:%(threadName)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logFMT = "%(asctime)s %(levelname)s %(name)s %(funcName)s() line:%(lineno)s] [%(message)s]"
logging.basicConfig(format=logFMT, level=loglevel)    
logger = logging.getLogger(notebookName)

In [2]:
#  setting the python path allows us to run python scripts from using
# # the CLI. 
ORIG_PYTHONPATH = os.environ['PYTHONPATH']

deconvolutionModules = notebookPath.parent.joinpath("../../../deconvolutionAnalysis/python/")
print("deconvolutionModules: {}\n".format(deconvolutionModules))

PYTHONPATH = ORIG_PYTHONPATH + f':{deconvolutionModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

intraExtraRNA_POCModules=notebookPath.parent.joinpath("../../python/src")
print("intraExtraRNA_POCModules: {}\n".format(intraExtraRNA_POCModules))

PYTHONPATH = PYTHONPATH + f':{intraExtraRNA_POCModules}'
print("PYTHONPATH: {}\n".format(PYTHONPATH))

########
os.environ["PYTHONPATH"] = PYTHONPATH
PYTHONPATH = os.environ["PYTHONPATH"]
print("PYTHONPATH: {}\n".format(PYTHONPATH))

# to be able to import our local python files we need to set the sys.path
# https://stackoverflow.com/a/50155834
sys.path.append( str(deconvolutionModules) )
sys.path.append( str(intraExtraRNA_POCModules) )
print("\nsys.path:\n{}\n".format(sys.path))

deconvolutionModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python

intraExtraRNA_POCModules: /private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../python/src

PYTHONPATH: :/private/home/aedavids/extraCellularRNA/src:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../../deconvolutionAnalysis/python:/private/home/aedavids/extraCellularRNA/intraExtraRNA_POC/jupyterNotebooks/elife/../../python/src


sys.p

In [3]:
# local packages
from analysis.utilities import loadList

In [4]:
dataRoot = "/private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data"
countDFPath = f'{dataRoot}/countDF.csv'
countDF = pd.read_csv(countDFPath, index_col='sample_id') # , index_col=
print(f'load {countDFPath}')

load /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/countDF.csv


In [5]:
print(f'countDF.shape : {countDF.shape}')

countDF.shape : (224, 76555)


In [6]:
countDF.iloc[0:3, 0:5]

,(A)n,(AAA)n,(AAAAAAC)n,(AAAAAAG)n,(AAAAAAT)n
sample_id,,,,,
SRR14506659,201.672053,0.0,0.0,0.0,0.0
SRR14506660,110.450773,0.0,0.0,0.0,0.0
SRR14506661,3722.776395,0.0,0.0,0.0,0.0


In [7]:
countDF.iloc[0:3, -5:]

,X8_LINE,X9_LINE,Zaphod,Zaphod2,Zaphod3
sample_id,,,,,
SRR14506659,0.0,0.0,170.645583,55.847645,24.821176
SRR14506660,0.0,0.0,0.000000,0.000000,0.000000
SRR14506661,0.0,0.0,0.000000,0.000000,0.000000


In [8]:
metaDFPath =  f'{dataRoot}/metaDF.csv'
metaDF = pd.read_csv(metaDFPath, index_col='sample_id')
print(f'load {metaDFPath}')

load /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/metaDF.csv


In [9]:
print(f'metaDF.shape : {metaDF.shape}')
metaDF.head()

metaDF.shape : (224, 1)


,diagnosis
sample_id,
SRR14506659,Esophagus Cancer
SRR14506660,Esophagus Cancer
SRR14506661,Esophagus Cancer
SRR14506662,Esophagus Cancer
SRR14506663,Esophagus Cancer


In [10]:
metaDF.loc[:, 'diagnosis'].unique()

array(['Esophagus Cancer', 'Lung Cancer', 'Liver Cancer',
       'Stomach Cancer', 'Colorectal Cancer', 'Healthy donor'],
      dtype=object)

In [11]:
selectLungIdRows = metaDF.loc[:, 'diagnosis'] == 'Lung Cancer'
lungSampleIds = metaDF.loc[ selectLungIdRows, :].index
lungSampleIds

Index(['SRR14506690', 'SRR14506691', 'SRR14506692', 'SRR14506693',
       'SRR14506694', 'SRR14506695', 'SRR14506696', 'SRR14506697',
       'SRR14506698', 'SRR14506699', 'SRR14506700', 'SRR14506701',
       'SRR14506702', 'SRR14506703', 'SRR14506704', 'SRR14506705',
       'SRR14506706', 'SRR14506707', 'SRR14506708', 'SRR14506709',
       'SRR14506710', 'SRR14506711', 'SRR14506712', 'SRR14506713',
       'SRR14506714', 'SRR14506715', 'SRR14506716', 'SRR14506717',
       'SRR14506718', 'SRR14506719', 'SRR14506720', 'SRR14506721',
       'SRR14506722', 'SRR14506723', 'SRR14506724'],
      dtype='object', name='sample_id')

In [12]:
selectHealthIdRows = metaDF.loc[:, 'diagnosis'] == 'Healthy donor'
healthySampleIds = metaDF.loc[ selectHealthIdRows, :].index
healthySampleIds

Index(['SRR14506843', 'SRR14506844', 'SRR14506845', 'SRR14506846',
       'SRR14506847', 'SRR14506848', 'SRR14506849', 'SRR14506850',
       'SRR14506851', 'SRR14506853', 'SRR14506854', 'SRR14506855',
       'SRR14506856', 'SRR14506857', 'SRR14506858', 'SRR14506859',
       'SRR14506860', 'SRR14506861', 'SRR14506862', 'SRR14506864',
       'SRR14506865', 'SRR14506866', 'SRR14506867', 'SRR14506868',
       'SRR14506869', 'SRR14506870', 'SRR14506871', 'SRR14506872',
       'SRR14506873', 'SRR14506874', 'SRR14506875', 'SRR14506877',
       'SRR14506878', 'SRR14506879', 'SRR14506880', 'SRR14506881',
       'SRR14506882', 'SRR14506883', 'SRR14506884', 'SRR14506885',
       'SRR14506886', 'SRR14506887', 'SRR14506888'],
      dtype='object', name='sample_id')

In [13]:
modelOut = "/private/groups/kimlab/aedavids/elife/elifeBinaryRandomForestResults.out/model"
modelName = "elife-Lung-Cancer-Health-control-random-forest-GTEx-Lung-biomarkers"
HUGOFeatureNamesPath = f"{modelOut}/{modelName}_features.txt"
HUGOFeatureNames = loadList(HUGOFeatureNamesPath)
HUGOFeatureNames

['FPR3',
 'CSF3',
 'SLAMF8',
 'ENTPD2',
 'MAGEE1',
 'PCAT19',
 'GRIP2',
 'PTGIR',
 'RND1',
 'CHRNB1']

In [19]:
mapDFPath =  f'{dataRoot}/mapDF.csv'
mapDF = pd.read_csv(mapDFPath, index_col='HUGO_v35')
print(f'load {mapDFPath}')

load /private/groups/kimlab/aedavids/elife/createGTEx_TCGA_elifeDataSets.out/data/mapDF.csv


In [20]:
print(f'mapDF.shape : {mapDF.shape}')
mapDF.head()

mapDF.shape : (70, 2)


,ENSG_v35,ENSG_v39
HUGO_v35,,
EBNA1BP2,ENSG00000117395.13,ENSG00000117395.13
SLAMF8,ENSG00000158714.11,ENSG00000158714.11
YOD1,ENSG00000180667.10,ENSG00000180667.11
SLC30A1,ENSG00000170385.10,ENSG00000170385.10
ROCK2,ENSG00000134318.14,ENSG00000134318.15


In [26]:
selectRows = mapDF.index.isin( HUGOFeatureNames )
sampleMapDF = mapDF.loc[ selectRows, ['ENSG_v39']]
display(sampleMapDF)
selectGeneIds = sampleMapDF.loc[:, 'ENSG_v39'].values
selectGeneIds

,ENSG_v39
HUGO_v35,
SLAMF8,ENSG00000158714.11
GRIP2,ENSG00000144596.13
ENTPD2,ENSG00000054179.12
RND1,ENSG00000172602.11
CHRNB1,ENSG00000170175.11
CSF3,ENSG00000108342.13
PCAT19,ENSG00000267107.9
PTGIR,ENSG00000160013.9
FPR3,ENSG00000187474.5


array(['ENSG00000158714.11', 'ENSG00000144596.13', 'ENSG00000054179.12',
       'ENSG00000172602.11', 'ENSG00000170175.11', 'ENSG00000108342.13',
       'ENSG00000267107.9', 'ENSG00000160013.9', 'ENSG00000187474.5',
       'ENSG00000198934.5'], dtype=object)

In [27]:
countDF.loc[selectLungIdRows, selectGeneIds]

,ENSG00000158714.11,ENSG00000144596.13,ENSG00000054179.12,ENSG00000172602.11,ENSG00000170175.11,ENSG00000108342.13,ENSG00000267107.9,ENSG00000160013.9,ENSG00000187474.5,ENSG00000198934.5
sample_id,,,,,,,,,,
SRR14506690,1.636995,3.455879,0.000000,2.182660,18.370724,0.363777,3.455879,135.870605,4.547209,1.818884
SRR14506691,0.000000,3.965234,0.000000,0.000000,13.217447,0.000000,10.573958,88.556897,0.000000,0.000000
SRR14506692,0.000000,0.000000,0.000000,9.293481,11.875003,0.516304,15.489135,106.358727,7.744567,6.195654
SRR14506693,0.000000,11.639525,0.000000,2.024265,12.651658,5.060663,28.339713,72.367482,0.000000,4.048530
SRR14506694,0.000000,1.241376,0.000000,0.413792,16.551686,0.000000,4.965506,153.930676,6.620674,15.310309
SRR14506695,3.118462,0.000000,0.623692,3.118462,13.721231,0.000000,11.226462,137.212314,4.365846,0.623692
SRR14506696,3.508385,0.438548,0.000000,1.315644,19.076845,0.657822,16.445556,126.959693,1.973467,3.289111
SRR14506697,4.863739,0.000000,0.000000,0.000000,39.604735,0.000000,21.539417,142.438083,0.000000,5.558559
SRR14506698,5.581392,0.000000,0.000000,0.000000,7.813949,0.000000,52.465085,49.116250,3.348835,0.000000


In [28]:
countDF.loc[healthySampleIds, selectGeneIds]

,ENSG00000158714.11,ENSG00000144596.13,ENSG00000054179.12,ENSG00000172602.11,ENSG00000170175.11,ENSG00000108342.13,ENSG00000267107.9,ENSG00000160013.9,ENSG00000187474.5,ENSG00000198934.5
sample_id,,,,,,,,,,
SRR14506843,1.835785,0.917893,0.000000,0.917893,5.507356,0.000000,267.106761,53.237774,55.073559,4.589463
SRR14506844,49.647848,11.127966,0.000000,0.000000,37.663884,0.000000,111.279658,92.447716,65.911798,17.119947
SRR14506845,0.000000,34.349732,0.000000,20.609839,16.487871,0.000000,142.894886,52.211593,54.959571,54.959571
SRR14506846,0.000000,0.000000,4.541199,0.000000,0.000000,0.000000,154.400761,27.247193,0.000000,0.000000
SRR14506847,0.000000,29.101215,0.000000,38.801620,58.202430,9.700405,9.700405,97.004050,0.000000,87.303645
SRR14506848,0.000000,0.000000,39.694282,0.000000,0.000000,0.000000,0.000000,92.619991,0.000000,0.000000
SRR14506849,23.005334,0.000000,0.000000,1.533689,44.476979,0.000000,113.492981,75.150758,65.948624,9.202134
SRR14506850,0.000000,8.259879,0.000000,1.835529,0.000000,0.000000,75.256672,10.095407,1.835529,14.684229
SRR14506851,0.000000,111.606062,0.000000,37.202021,65.103536,0.000000,0.000000,144.157831,0.000000,0.000000
